In [0]:
# ============================================================================
# PRICING FEATURE ADOPTION - PRODUCT HEALTH BASELINE
# ============================================================================
# Framework: Product Director Dashboard
# Focus: ADOPTION METRICS (Reach, Activation, Engagement Depth)
# Hierarchy: GMV Tier → Supplier Segment → Activity Type → Features
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings('ignore')

# Professional color palettes
COLORS = {
    'gmv_tiers': ['#E3F2FD', '#90CAF9', '#42A5F5', '#1565C0'],  # Light to Dark Blue
    'features': ['#FFE0B2', '#FFB74D', '#FF9800', '#E65100', '#BF360C'],  # Amber/Orange gradient
    'segments': ['#F5F5F5', '#BDBDBD', '#757575', '#424242'],  # Grey gradient
    'heatmap_primary': 'YlOrRd',  # Yellow-Orange-Red for primary heatmaps
    'heatmap_secondary': 'Blues',  # Blues for secondary heatmaps
    'positive': '#2E7D32',  # Dark green for positive metrics
    'negative': '#C62828',  # Dark red for negative/opportunity
    'neutral': '#757575'   # Grey for neutral
}

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.family'] = 'sans-serif'

print("="*100)
print("PRICING FEATURES - ADOPTION BASELINE ANALYSIS")
print("Product Health Dashboard Framework")
print("="*100)
print(f"Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d')}")
print("="*100)


In [0]:
# ============================================================================
# PRICING FEATURES - FEATURE ADOPTION ANALYSIS (CLEAN VERSION)
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings('ignore')

# Clean color palette: Orange gradient
COLORS = {
    'light': '#FFE0B2',
    'medium_light': '#FFB74D', 
    'medium': '#FF9800',
    'medium_dark': '#F57C00',
    'dark': '#E65100',
    'grey_light': '#EEEEEE',
    'grey': '#BDBDBD',
    'grey_dark': '#757575',
    'border': '#424242'
}

print("="*100)
print("PRICING FEATURES - FEATURE ADOPTION ANALYSIS")
print("="*100)

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================

print("\n[1/4] Loading data...")

df_spark = spark.table("production.supply_analytics.pricing_feature_combined")

# Create supplier GMV tiers
supplier_gmv = df_spark.filter(F.col("gmv_l12mo") > 0) \
    .groupBy("supplier_id") \
    .agg(F.sum("gmv_l12mo").alias("supplier_gmv_l12mo"))

window_spec = Window.orderBy("supplier_gmv_l12mo")
supplier_gmv = supplier_gmv.withColumn("gmv_quartile", F.ntile(4).over(window_spec)) \
    .withColumn("gmv_tier",
        F.when(F.col("gmv_quartile") == 1, F.lit("Bottom 25%"))
         .when(F.col("gmv_quartile") == 2, F.lit("25-50%"))
         .when(F.col("gmv_quartile") == 3, F.lit("50-75%"))
         .when(F.col("gmv_quartile") == 4, F.lit("Top 25%"))
    )

df_spark = df_spark.join(
    supplier_gmv.select("supplier_id", F.col("gmv_tier")),
    on="supplier_id", how="left"
)

df = df_spark.toPandas()

# Convert decimals
decimal_cols = ['gmv_l12mo', 'nr_l12mo', 'bookings_l12mo']
for col in decimal_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Filter active tours only
df = df[df['gmv_l12mo'] > 0].copy()
df['supplier_segment'] = df['supplier_segment'].fillna('Unknown')
df['activity_type'] = df['activity_type'].fillna('Unknown')

# FIX: Convert is_managed properly (handles 'Yes'/'No' strings)
if 'is_managed' in df.columns:
    df['is_managed'] = df['is_managed'].apply(lambda x: 1 if x in [1, 'Yes', True, '1'] else 0)
else:
    df['is_managed'] = 0

# Define features to analyze (ONLY the actual advanced features)
PRICING_FEATURES = {
    'has_group_pricing': 'Group Pricing',
    'has_addons': 'Add-ons',
    'has_native_scales': 'Native Pricing Scales',
    'has_api_scales': 'API Pricing Scales',
    'has_static_api_pricing': 'Static API Pricing',
    'has_live_dynamic_pricing': 'Live Dynamic Pricing'
}

# Individual pricing categories (subset of tours WITH individual pricing)
INDIVIDUAL_CATEGORIES = {
    'has_adult_category': 'Adult',
    'has_child_category': 'Child',
    'has_infant_category': 'Infant',
    'has_senior_category': 'Senior',
    'has_youth_student_category': 'Youth/Student'
}

GMV_TIERS = ['Bottom 25%', '25-50%', '50-75%', 'Top 25%']

print(f"✓ Loaded: {len(df):,} tours, {df['supplier_id'].nunique():,} suppliers")
print(f"✓ GMV: €{df['gmv_l12mo'].sum()/1e6:.1f}M")

# ============================================================================
# Print segmentation explanations (NOT on charts)
# ============================================================================

print("\n" + "="*100)
print("SEGMENTATION GUIDE")
print("="*100)
print("\nGMV Tier: Supplier revenue quartile (Bottom 25%, 25-50%, 50-75%, Top 25%)")
print("  - Based on supplier total GMV last 12 months")
print("  - Equal number of suppliers per tier")
print("\nSupplier Segment: Business classification")
print("  - Scale Seeker, Independent Creator, Heritage Preserver, Leisure Brand, etc.")
print("\nActivity Type: Primary experience category")
print("  - privateTour, transfer, dayTrip, waterActivity, adventure, guidedTour, etc.")
print("\nManaged Status: Whether supplier has dedicated account manager")
print("  - Managed: Yes | Not Managed: No")
print("="*100 + "\n")

# ============================================================================
# STEP 2: OVERALL FEATURE ADOPTION
# ============================================================================

print("\n[2/4] Overall feature adoption rates...")

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle('PRICING FEATURES - OVERALL ADOPTION RATES', fontsize=18, fontweight='bold', y=1.02)

# Chart 1: Advanced Features Activation
ax1 = axes[0]
activation_rates = {}
for col, name in PRICING_FEATURES.items():
    if col in df.columns:
        activation_rates[name] = 100 * df[col].sum() / len(df)

sorted_features = sorted(activation_rates.items(), key=lambda x: x[1], reverse=True)
names = [x[0] for x in sorted_features]
rates = [x[1] for x in sorted_features]

# Color gradient: darker orange = higher adoption
colors = [COLORS['dark'] if r > 30 else COLORS['medium_dark'] if r > 20 else 
          COLORS['medium'] if r > 10 else COLORS['medium_light'] for r in rates]

bars = ax1.barh(range(len(names)), rates, color=colors, alpha=0.85, 
                edgecolor=COLORS['border'], linewidth=1.5)
ax1.set_yticks(range(len(names)))
ax1.set_yticklabels(names, fontsize=11)
ax1.set_xlabel('Activation Rate (%)', fontweight='bold', fontsize=12)
ax1.set_title('Advanced Pricing Features', fontweight='bold', fontsize=14, pad=15)
ax1.grid(axis='x', alpha=0.2, color=COLORS['grey_dark'])
ax1.set_xlim(0, max(rates) * 1.25)

for i, (name, rate) in enumerate(sorted_features):
    col_name = [k for k, v in PRICING_FEATURES.items() if v == name][0]
    tours = df[col_name].sum()
    ax1.text(rate + max(rates)*0.02, i, f'{rate:.1f}%\n({tours:,})', 
            va='center', fontsize=10, fontweight='bold')

# Chart 2: Individual Pricing Categories (among tours WITH individual pricing)
ax2 = axes[1]
ind_base = df[df['has_individual_pricing'] == 1] if 'has_individual_pricing' in df.columns else df
category_rates = {}
for col, name in INDIVIDUAL_CATEGORIES.items():
    if col in ind_base.columns:
        category_rates[name] = 100 * ind_base[col].sum() / len(ind_base)

sorted_cats = sorted(category_rates.items(), key=lambda x: x[1], reverse=True)
cat_names = [x[0] for x in sorted_cats]
cat_rates = [x[1] for x in sorted_cats]

colors_cat = [COLORS['dark'] if r > 80 else COLORS['medium_dark'] if r > 60 else 
              COLORS['medium'] if r > 40 else COLORS['medium_light'] for r in cat_rates]

bars = ax2.barh(range(len(cat_names)), cat_rates, color=colors_cat, alpha=0.85,
                edgecolor=COLORS['border'], linewidth=1.5)
ax2.set_yticks(range(len(cat_names)))
ax2.set_yticklabels(cat_names, fontsize=11)
ax2.set_xlabel('% of Tours with Individual Pricing', fontweight='bold', fontsize=12)
ax2.set_title('Individual Pricing Categories', fontweight='bold', fontsize=14, pad=15)
ax2.grid(axis='x', alpha=0.2, color=COLORS['grey_dark'])
ax2.set_xlim(0, 105)

for i, (name, rate) in enumerate(sorted_cats):
    ax2.text(rate + 2, i, f'{rate:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Overall adoption charts created")

# ============================================================================
# STEP 3: ADOPTION BY GMV TIER
# ============================================================================

print("\n[3/4] Adoption by GMV tier...")

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
fig.suptitle('FEATURE ADOPTION BY GMV TIER', fontsize=18, fontweight='bold', y=1.00)

# Chart 1: Heatmap - Features × GMV Tier
ax1 = axes[0]
heatmap_data = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in tier_df.columns:
            rate = 100 * tier_df[col].sum() / len(tier_df) if len(tier_df) > 0 else 0
            row.append(rate)
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data, columns=list(PRICING_FEATURES.values()), index=GMV_TIERS)

sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax1,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=60, annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax1.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax1.set_ylabel('GMV Tier', fontweight='bold', fontsize=13)
ax1.set_title('Activation Rate by GMV Tier (%)', fontweight='bold', fontsize=14, pad=15)
ax1.set_yticklabels(GMV_TIERS, rotation=0, fontsize=11)
ax1.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=10)

# Chart 2: Grouped bars - Comparison across tiers
ax2 = axes[1]
x = np.arange(len(GMV_TIERS))
width = 0.15

# Select top 4 features to avoid clutter
top_features = sorted_features[:4]
colors_bar = [COLORS['dark'], COLORS['medium_dark'], COLORS['medium'], COLORS['medium_light']]

for i, (feat_name, _) in enumerate(top_features):
    feat_col = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
    rates_by_tier = []
    for tier in GMV_TIERS:
        tier_df = df[df['gmv_tier'] == tier]
        rate = 100 * tier_df[feat_col].sum() / len(tier_df) if len(tier_df) > 0 else 0
        rates_by_tier.append(rate)
    
    offset = width * (i - 1.5)
    ax2.bar(x + offset, rates_by_tier, width, label=feat_name, 
            color=colors_bar[i], alpha=0.85, edgecolor=COLORS['border'], linewidth=1.2)

ax2.set_xticks(x)
ax2.set_xticklabels(GMV_TIERS, fontsize=11)
ax2.set_ylabel('Activation Rate (%)', fontweight='bold', fontsize=12)
ax2.set_xlabel('GMV Tier', fontweight='bold', fontsize=12)
ax2.set_title('Top Features by GMV Tier', fontweight='bold', fontsize=14, pad=15)
ax2.legend(loc='upper left', fontsize=10, framealpha=0.95)
ax2.grid(axis='y', alpha=0.2, color=COLORS['grey_dark'])

plt.tight_layout()
plt.show()

print("✓ GMV tier charts created")

# ============================================================================
# STEP 4: ADOPTION BY SEGMENT & ACTIVITY TYPE
# ============================================================================

print("\n[4/4] Adoption by segment and activity type...")

fig, axes = plt.subplots(2, 1, figsize=(22, 14))
fig.suptitle('FEATURE ADOPTION BY SUPPLIER SEGMENT & ACTIVITY TYPE', 
             fontsize=18, fontweight='bold', y=0.995)

# Chart 1: By Supplier Segment
ax1 = axes[0]
top_segments = df['supplier_segment'].value_counts().head(8).index.tolist()
heatmap_seg = []
seg_labels = []
for seg in top_segments:
    seg_df = df[df['supplier_segment'] == seg]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in seg_df.columns:
            rate = 100 * seg_df[col].sum() / len(seg_df) if len(seg_df) > 0 else 0
            row.append(rate)
    heatmap_seg.append(row)
    seg_labels.append(f"{seg} (n={len(seg_df):,})")

heatmap_seg_df = pd.DataFrame(heatmap_seg, columns=list(PRICING_FEATURES.values()), index=seg_labels)

sns.heatmap(heatmap_seg_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax1,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=70, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax1.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax1.set_ylabel('Supplier Segment', fontweight='bold', fontsize=13)
ax1.set_title('Activation Rate by Supplier Segment (%)', fontweight='bold', fontsize=15, pad=15)
ax1.set_yticklabels(seg_labels, rotation=0, fontsize=11)
ax1.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=11)

# Chart 2: By Activity Type
ax2 = axes[1]
top_activities = df['activity_type'].value_counts().head(10).index.tolist()
heatmap_act = []
act_labels = []
for act in top_activities:
    act_df = df[df['activity_type'] == act]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in act_df.columns:
            rate = 100 * act_df[col].sum() / len(act_df) if len(act_df) > 0 else 0
            row.append(rate)
    heatmap_act.append(row)
    act_labels.append(f"{act} (n={len(act_df):,})")

heatmap_act_df = pd.DataFrame(heatmap_act, columns=list(PRICING_FEATURES.values()), index=act_labels)

sns.heatmap(heatmap_act_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax2,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=80, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax2.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax2.set_ylabel('Activity Type', fontweight='bold', fontsize=13)
ax2.set_title('Activation Rate by Activity Type (%)', fontweight='bold', fontsize=15, pad=15)
ax2.set_yticklabels(act_labels, rotation=0, fontsize=10)
ax2.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=11)

plt.tight_layout()
plt.show()

print("✓ Segment & activity charts created")

# ============================================================================
# PRINT SUMMARY STATS
# ============================================================================

print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)

print("\nOVERALL ADOPTION:")
for feat_name, rate in sorted_features:
    col_name = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
    tours = df[col_name].sum()
    print(f"  {feat_name}: {rate:.1f}% ({tours:,} tours)")

print("\nBY GMV TIER:")
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    print(f"\n  {tier}: {len(tier_df):,} tours ({100*len(tier_df)/len(df):.1f}%)")
    for feat_name, _ in sorted_features[:3]:
        col_name = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
        rate = 100 * tier_df[col_name].sum() / len(tier_df)
        print(f"    - {feat_name}: {rate:.1f}%")

print("\nBY MANAGED STATUS:")
for managed in [0, 1]:
    managed_df = df[df['is_managed'] == managed]
    status = "Managed" if managed == 1 else "Not Managed"
    print(f"\n  {status}: {len(managed_df):,} tours ({100*len(managed_df)/len(df):.1f}%)")
    for feat_name, _ in sorted_features[:3]:
        col_name = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
        rate = 100 * managed_df[col_name].sum() / len(managed_df)
        print(f"    - {feat_name}: {rate:.1f}%")

print("\n" + "="*100)
print("✓ FEATURE ADOPTION ANALYSIS COMPLETE")
print("="*100)


In [0]:
# ============================================================================
# PRICING FEATURES - FEATURE ADOPTION ANALYSIS (CLEAN VERSION)
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings('ignore')

# Clean color palette: Orange gradient
COLORS = {
    'light': '#FFE0B2',
    'medium_light': '#FFB74D', 
    'medium': '#FF9800',
    'medium_dark': '#F57C00',
    'dark': '#E65100',
    'grey_light': '#EEEEEE',
    'grey': '#BDBDBD',
    'grey_dark': '#757575',
    'border': '#424242'
}

print("="*100)
print("PRICING FEATURES - FEATURE ADOPTION ANALYSIS")
print("="*100)

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================

print("\n[1/4] Loading data...")

df_spark = spark.table("production.supply_analytics.pricing_feature_combined")

# Create supplier GMV tiers
supplier_gmv = df_spark.filter(F.col("gmv_l12mo") > 0) \
    .groupBy("supplier_id") \
    .agg(F.sum("gmv_l12mo").alias("supplier_gmv_l12mo"))

window_spec = Window.orderBy("supplier_gmv_l12mo")
supplier_gmv = supplier_gmv.withColumn("gmv_quartile", F.ntile(4).over(window_spec)) \
    .withColumn("gmv_tier",
        F.when(F.col("gmv_quartile") == 1, F.lit("Bottom 25%"))
         .when(F.col("gmv_quartile") == 2, F.lit("25-50%"))
         .when(F.col("gmv_quartile") == 3, F.lit("50-75%"))
         .when(F.col("gmv_quartile") == 4, F.lit("Top 25%"))
    )

df_spark = df_spark.join(
    supplier_gmv.select("supplier_id", F.col("gmv_tier")),
    on="supplier_id", how="left"
)

df = df_spark.toPandas()

# Convert decimals
decimal_cols = ['gmv_l12mo', 'nr_l12mo', 'bookings_l12mo']
for col in decimal_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Filter active tours only
df = df[df['gmv_l12mo'] > 0].copy()
df['supplier_segment'] = df['supplier_segment'].fillna('Unknown')
df['activity_type'] = df['activity_type'].fillna('Unknown')
df['is_managed'] = df['is_managed'].fillna('No').map({'Yes': 1, 'No': 0})

# Define features to analyze (ONLY the actual advanced features)
PRICING_FEATURES = {
    'has_group_pricing': 'Group Pricing',
    'has_addons': 'Add-ons',
    'has_native_scales': 'Native Pricing Scales',
    'has_api_scales': 'API Pricing Scales',
    'has_static_api_pricing': 'Static API Pricing',
    'has_live_dynamic_pricing': 'Live Dynamic Pricing'
}

# Individual pricing categories (subset of tours WITH individual pricing)
INDIVIDUAL_CATEGORIES = {
    'has_adult_category': 'Adult',
    'has_child_category': 'Child',
    'has_infant_category': 'Infant',
    'has_senior_category': 'Senior',
    'has_youth_student_category': 'Youth/Student'
}

GMV_TIERS = ['Bottom 25%', '25-50%', '50-75%', 'Top 25%']

print(f"✓ Loaded: {len(df):,} tours, {df['supplier_id'].nunique():,} suppliers")
print(f"✓ GMV: €{df['gmv_l12mo'].sum()/1e6:.1f}M")

# ============================================================================
# Print segmentation explanations (NOT on charts)
# ============================================================================

print("\n" + "="*100)
print("SEGMENTATION GUIDE")
print("="*100)
print("\nGMV Tier: Supplier revenue quartile (Bottom 25%, 25-50%, 50-75%, Top 25%)")
print("  - Based on supplier total GMV last 12 months")
print("  - Equal number of suppliers per tier")
print("\nSupplier Segment: Business classification")
print("  - Scale Seeker, Independent Creator, Heritage Preserver, Leisure Brand, etc.")
print("\nActivity Type: Primary experience category")
print("  - privateTour, transfer, dayTrip, waterActivity, adventure, guidedTour, etc.")
print("\nManaged Status: Whether supplier has dedicated account manager")
print("  - Managed: Yes | Not Managed: No")
print("="*100 + "\n")

# ============================================================================
# STEP 2: OVERALL FEATURE ADOPTION
# ============================================================================

print("\n[2/4] Overall feature adoption rates...")

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle('PRICING FEATURES - OVERALL ADOPTION RATES', fontsize=18, fontweight='bold', y=1.02)

# Chart 1: Advanced Features Activation
ax1 = axes[0]
activation_rates = {}
for col, name in PRICING_FEATURES.items():
    if col in df.columns:
        activation_rates[name] = 100 * df[col].sum() / len(df)

sorted_features = sorted(activation_rates.items(), key=lambda x: x[1], reverse=True)
names = [x[0] for x in sorted_features]
rates = [x[1] for x in sorted_features]

# Color gradient: darker orange = higher adoption
colors = [COLORS['dark'] if r > 30 else COLORS['medium_dark'] if r > 20 else 
          COLORS['medium'] if r > 10 else COLORS['medium_light'] for r in rates]

bars = ax1.barh(range(len(names)), rates, color=colors, alpha=0.85, 
                edgecolor=COLORS['border'], linewidth=1.5)
ax1.set_yticks(range(len(names)))
ax1.set_yticklabels(names, fontsize=11)
ax1.set_xlabel('Activation Rate (%)', fontweight='bold', fontsize=12)
ax1.set_title('Advanced Pricing Features', fontweight='bold', fontsize=14, pad=15)
ax1.grid(axis='x', alpha=0.2, color=COLORS['grey_dark'])
ax1.set_xlim(0, max(rates) * 1.25)

for i, (name, rate) in enumerate(sorted_features):
    col_name = [k for k, v in PRICING_FEATURES.items() if v == name][0]
    tours = df[col_name].sum()
    ax1.text(rate + max(rates)*0.02, i, f'{rate:.1f}%\n({tours:,})', 
            va='center', fontsize=10, fontweight='bold')

# Chart 2: Individual Pricing Categories (among tours WITH individual pricing)
ax2 = axes[1]
ind_base = df[df['has_individual_pricing'] == 1] if 'has_individual_pricing' in df.columns else df
category_rates = {}
for col, name in INDIVIDUAL_CATEGORIES.items():
    if col in ind_base.columns:
        category_rates[name] = 100 * ind_base[col].sum() / len(ind_base)

sorted_cats = sorted(category_rates.items(), key=lambda x: x[1], reverse=True)
cat_names = [x[0] for x in sorted_cats]
cat_rates = [x[1] for x in sorted_cats]

colors_cat = [COLORS['dark'] if r > 80 else COLORS['medium_dark'] if r > 60 else 
              COLORS['medium'] if r > 40 else COLORS['medium_light'] for r in cat_rates]

bars = ax2.barh(range(len(cat_names)), cat_rates, color=colors_cat, alpha=0.85,
                edgecolor=COLORS['border'], linewidth=1.5)
ax2.set_yticks(range(len(cat_names)))
ax2.set_yticklabels(cat_names, fontsize=11)
ax2.set_xlabel('% of Tours with Individual Pricing', fontweight='bold', fontsize=12)
ax2.set_title('Individual Pricing Categories', fontweight='bold', fontsize=14, pad=15)
ax2.grid(axis='x', alpha=0.2, color=COLORS['grey_dark'])
ax2.set_xlim(0, 105)

for i, (name, rate) in enumerate(sorted_cats):
    ax2.text(rate + 2, i, f'{rate:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Overall adoption charts created")

# ============================================================================
# STEP 3: ADOPTION BY GMV TIER
# ============================================================================

print("\n[3/4] Adoption by GMV tier...")

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
fig.suptitle('FEATURE ADOPTION BY GMV TIER', fontsize=18, fontweight='bold', y=1.00)

# Chart 1: Heatmap - Features × GMV Tier
ax1 = axes[0]
heatmap_data = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in tier_df.columns:
            rate = 100 * tier_df[col].sum() / len(tier_df) if len(tier_df) > 0 else 0
            row.append(rate)
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data, columns=list(PRICING_FEATURES.values()), index=GMV_TIERS)

sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax1,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=60, annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax1.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax1.set_ylabel('GMV Tier', fontweight='bold', fontsize=13)
ax1.set_title('Activation Rate by GMV Tier (%)', fontweight='bold', fontsize=14, pad=15)
ax1.set_yticklabels(GMV_TIERS, rotation=0, fontsize=11)
ax1.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=10)

# Chart 2: Grouped bars - Comparison across tiers
ax2 = axes[1]
x = np.arange(len(GMV_TIERS))
width = 0.15

# Select top 4 features to avoid clutter
top_features = sorted_features[:4]
colors_bar = [COLORS['dark'], COLORS['medium_dark'], COLORS['medium'], COLORS['medium_light']]

for i, (feat_name, _) in enumerate(top_features):
    feat_col = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
    rates_by_tier = []
    for tier in GMV_TIERS:
        tier_df = df[df['gmv_tier'] == tier]
        rate = 100 * tier_df[feat_col].sum() / len(tier_df) if len(tier_df) > 0 else 0
        rates_by_tier.append(rate)
    
    offset = width * (i - 1.5)
    ax2.bar(x + offset, rates_by_tier, width, label=feat_name, 
            color=colors_bar[i], alpha=0.85, edgecolor=COLORS['border'], linewidth=1.2)

ax2.set_xticks(x)
ax2.set_xticklabels(GMV_TIERS, fontsize=11)
ax2.set_ylabel('Activation Rate (%)', fontweight='bold', fontsize=12)
ax2.set_xlabel('GMV Tier', fontweight='bold', fontsize=12)
ax2.set_title('Top Features by GMV Tier', fontweight='bold', fontsize=14, pad=15)
ax2.legend(loc='upper left', fontsize=10, framealpha=0.95)
ax2.grid(axis='y', alpha=0.2, color=COLORS['grey_dark'])

plt.tight_layout()
plt.show()

print("✓ GMV tier charts created")

# ============================================================================
# STEP 4: ADOPTION BY SEGMENT & ACTIVITY TYPE
# ============================================================================

print("\n[4/4] Adoption by segment and activity type...")

fig, axes = plt.subplots(2, 1, figsize=(22, 14))
fig.suptitle('FEATURE ADOPTION BY SUPPLIER SEGMENT & ACTIVITY TYPE', 
             fontsize=18, fontweight='bold', y=0.995)

# Chart 1: By Supplier Segment
ax1 = axes[0]
top_segments = df['supplier_segment'].value_counts().head(8).index.tolist()
heatmap_seg = []
seg_labels = []
for seg in top_segments:
    seg_df = df[df['supplier_segment'] == seg]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in seg_df.columns:
            rate = 100 * seg_df[col].sum() / len(seg_df) if len(seg_df) > 0 else 0
            row.append(rate)
    heatmap_seg.append(row)
    seg_labels.append(f"{seg} (n={len(seg_df):,})")

heatmap_seg_df = pd.DataFrame(heatmap_seg, columns=list(PRICING_FEATURES.values()), index=seg_labels)

sns.heatmap(heatmap_seg_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax1,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=70, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax1.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax1.set_ylabel('Supplier Segment', fontweight='bold', fontsize=13)
ax1.set_title('Activation Rate by Supplier Segment (%)', fontweight='bold', fontsize=15, pad=15)
ax1.set_yticklabels(seg_labels, rotation=0, fontsize=11)
ax1.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=11)

# Chart 2: By Activity Type
ax2 = axes[1]
top_activities = df['activity_type'].value_counts().head(10).index.tolist()
heatmap_act = []
act_labels = []
for act in top_activities:
    act_df = df[df['activity_type'] == act]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in act_df.columns:
            rate = 100 * act_df[col].sum() / len(act_df) if len(act_df) > 0 else 0
            row.append(rate)
    heatmap_act.append(row)
    act_labels.append(f"{act} (n={len(act_df):,})")

heatmap_act_df = pd.DataFrame(heatmap_act, columns=list(PRICING_FEATURES.values()), index=act_labels)

sns.heatmap(heatmap_act_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax2,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=80, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax2.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax2.set_ylabel('Activity Type', fontweight='bold', fontsize=13)
ax2.set_title('Activation Rate by Activity Type (%)', fontweight='bold', fontsize=15, pad=15)
ax2.set_yticklabels(act_labels, rotation=0, fontsize=10)
ax2.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=11)

plt.tight_layout()
plt.show()

print("✓ Segment & activity charts created")

# ============================================================================
# PRINT SUMMARY STATS
# ============================================================================

print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)

print("\nOVERALL ADOPTION:")
for feat_name, rate in sorted_features:
    col_name = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
    tours = df[col_name].sum()
    print(f"  {feat_name}: {rate:.1f}% ({tours:,} tours)")

print("\nBY GMV TIER:")
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    print(f"\n  {tier}: {len(tier_df):,} tours ({100*len(tier_df)/len(df):.1f}%)")
    for feat_name, _ in sorted_features[:3]:
        col_name = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
        rate = 100 * tier_df[col_name].sum() / len(tier_df)
        print(f"    - {feat_name}: {rate:.1f}%")

print("\nBY MANAGED STATUS:")
for managed in [0, 1]:
    managed_df = df[df['is_managed'] == managed]
    status = "Managed" if managed == 1 else "Not Managed"
    print(f"\n  {status}: {len(managed_df):,} tours ({100*len(managed_df)/len(df):.1f}%)")
    for feat_name, _ in sorted_features[:3]:
        col_name = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
        rate = 100 * managed_df[col_name].sum() / len(managed_df)
        print(f"    - {feat_name}: {rate:.1f}%")

print("\n" + "="*100)
print("✓ FEATURE ADOPTION ANALYSIS COMPLETE")
print("="*100)


In [0]:
# ============================================================================
# STEP 3: ACTIVATION & ENGAGEMENT BY GMV TIER
# ============================================================================

print("\n[3/5] Creating GMV tier deep dives...")

for tier_idx, tier in enumerate(GMV_TIERS):
    tier_df = df[df['gmv_tier'] == tier].copy()
    
    if len(tier_df) == 0:
        continue
    
    print(f"\n  Analyzing {tier}: {len(tier_df):,} tours, {tier_df['supplier_id'].nunique():,} suppliers")
    
    fig = plt.figure(figsize=(24, 16))
    gs = fig.add_gridspec(4, 3, hspace=0.38, wspace=0.32, top=0.94, bottom=0.06)
    
    tier_color = COLORS['gmv_tiers'][tier_idx]
    
    fig.suptitle(f'{tier.upper()} - ADOPTION BASELINE\nREACH: {len(tier_df):,} tours | {tier_df["supplier_id"].nunique():,} suppliers | €{tier_df["gmv_l12mo"].sum()/1e6:.1f}M GMV',
                fontsize=20, fontweight='bold', y=0.97)
    
    # ========== ROW 1: ACTIVATION METRICS ==========
    
    # Chart 1.1: Feature Activation Rates
    ax1 = fig.add_subplot(gs[0, 0])
    tier_activation = {}
    for col, name in PRICING_FEATURES.items():
        if col in tier_df.columns:
            tier_activation[name] = 100 * tier_df[col].sum() / len(tier_df)
    
    sorted_act = sorted(tier_activation.items(), key=lambda x: x[1], reverse=True)
    feat_names = [x[0] for x in sorted_act]
    feat_rates = [x[1] for x in sorted_act]
    
    bars = ax1.barh(range(len(feat_names)), feat_rates,
                   color=COLORS['features'][:len(feat_names)], alpha=0.85, 
                   edgecolor='#424242', linewidth=1.2)
    ax1.set_yticks(range(len(feat_names)))
    ax1.set_yticklabels(feat_names, fontsize=10)
    ax1.set_xlabel('Activation Rate (%)', fontweight='bold', fontsize=11)
    ax1.set_title('ACTIVATION: Feature Adoption', fontweight='bold', fontsize=13, pad=12)
    ax1.grid(axis='x', alpha=0.25)
    
    for i, (name, rate) in enumerate(sorted_act):
        tours = tier_df[[k for k, v in PRICING_FEATURES.items() if v == name][0]].sum()
        ax1.text(rate + max(feat_rates)*0.02, i, f'{rate:.1f}%\n({tours:,})', 
                va='center', fontsize=9, fontweight='bold')
    
    # Chart 1.2: Engagement Depth
    ax2 = fig.add_subplot(gs[0, 1])
    soph_dist = tier_df['feature_sophistication'].value_counts().reindex([
        'Basic (Individual Only)',
        'Single Advanced Feature',
        'Multi-Feature',
        'Advanced',
        'Power User'
    ], fill_value=0)
    
    colors_soph = ['#BDBDBD', '#FFE0B2', '#FFB74D', '#FF9800', '#E65100']
    bars = ax2.bar(range(len(soph_dist)), soph_dist.values,
                  color=colors_soph, alpha=0.85, edgecolor='#424242', linewidth=1.2)
    ax2.set_xticks(range(len(soph_dist)))
    ax2.set_xticklabels(['Basic', 'Single', 'Multi', 'Advanced', 'Power'], fontsize=9, rotation=20, ha='right')
    ax2.set_ylabel('Number of Tours', fontweight='bold', fontsize=11)
    ax2.set_title('ENGAGEMENT DEPTH: Sophistication', fontweight='bold', fontsize=13, pad=12)
    ax2.grid(axis='y', alpha=0.25)
    
    for i, count in enumerate(soph_dist.values):
        pct = 100 * count / len(tier_df)
        ax2.text(i, count + count*0.02, f'{count:,}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Chart 1.3: Managed vs Not Managed
    ax3 = fig.add_subplot(gs[0, 2])
    managed_breakdown = tier_df.groupby('is_managed').agg({
        'tour_id': 'count',
        'num_core_features': lambda x: (x > 0).sum()
    })
    managed_breakdown.index = ['Not Managed', 'Managed']
    managed_breakdown['rate'] = 100 * managed_breakdown['num_core_features'] / managed_breakdown['tour_id']
    
    bars = ax3.bar(['Not Managed', 'Managed'], managed_breakdown['rate'],
                  color=[COLORS['segments'][2], tier_color], alpha=0.85, 
                  edgecolor='#424242', linewidth=1.2)
    ax3.set_ylabel('Activation Rate (%)', fontweight='bold', fontsize=11)
    ax3.set_title('FILTER: Managed Status', fontweight='bold', fontsize=13, pad=12)
    ax3.grid(axis='y', alpha=0.25)
    
    for i, (idx, row) in enumerate(managed_breakdown.iterrows()):
        ax3.text(i, row['rate'] + 1, f"{row['rate']:.1f}%\n({row['tour_id']:,})",
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # ========== ROW 2: BY SUPPLIER SEGMENT ==========
    
    ax4 = fig.add_subplot(gs[1, :])
    top_segs = tier_df['supplier_segment'].value_counts().head(10).index.tolist()
    heatmap_seg = []
    seg_labels = []
    for seg in top_segs:
        seg_df = tier_df[tier_df['supplier_segment'] == seg]
        row = []
        for col in PRICING_FEATURES.keys():
            if col in seg_df.columns:
                adoption = 100 * seg_df[col].sum() / len(seg_df) if len(seg_df) > 0 else 0
                row.append(adoption)
        heatmap_seg.append(row)
        seg_labels.append(f"{seg}\n(n={len(seg_df):,})")
    
    heatmap_seg_df = pd.DataFrame(heatmap_seg, columns=list(PRICING_FEATURES.values()), index=seg_labels)
    sns.heatmap(heatmap_seg_df, annot=True, fmt='.1f', cmap=COLORS['heatmap_primary'], ax=ax4,
                cbar_kws={'label': 'Activation %'}, linewidths=2, linecolor='white', 
                vmin=0, vmax=70, annot_kws={'fontsize': 9})
    ax4.set_xlabel('Pricing Features', fontweight='bold', fontsize=12)
    ax4.set_ylabel('Supplier Segment', fontweight='bold', fontsize=12)
    ax4.set_title('ACTIVATION BY SUPPLIER SEGMENT (% of tours in segment)', fontweight='bold', fontsize=14, pad=15)
    ax4.set_yticklabels(seg_labels, rotation=0, fontsize=9)
    ax4.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=10)
    
    # ========== ROW 3: BY ACTIVITY TYPE ==========
    
    ax5 = fig.add_subplot(gs[2, :])
    top_acts = tier_df['activity_type'].value_counts().head(10).index.tolist()
    heatmap_act = []
    act_labels = []
    for act in top_acts:
        act_df = tier_df[tier_df['activity_type'] == act]
        row = []
        for col in PRICING_FEATURES.keys():
            if col in act_df.columns:
                adoption = 100 * act_df[col].sum() / len(act_df) if len(act_df) > 0 else 0
                row.append(adoption)
        heatmap_act.append(row)
        act_labels.append(f"{act}\n(n={len(act_df):,})")
    
    heatmap_act_df = pd.DataFrame(heatmap_act, columns=list(PRICING_FEATURES.values()), index=act_labels)
    sns.heatmap(heatmap_act_df, annot=True, fmt='.1f', cmap=COLORS['heatmap_secondary'], ax=ax5,
                cbar_kws={'label': 'Activation %'}, linewidths=2, linecolor='white',
                vmin=0, vmax=70, annot_kws={'fontsize': 9})
    ax5.set_xlabel('Pricing Features', fontweight='bold', fontsize=12)
    ax5.set_ylabel('Activity Type', fontweight='bold', fontsize=12)
    ax5.set_title('ACTIVATION BY ACTIVITY TYPE (% of tours in category)', fontweight='bold', fontsize=14, pad=15)
    ax5.set_yticklabels(act_labels, rotation=0, fontsize=9)
    ax5.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=10)
    
    # ========== ROW 4: SUMMARY STATS ==========
    
    # Chart 4.1: Top Segment × Activity Combinations
    ax6 = fig.add_subplot(gs[3, 0])
    combo = tier_df.groupby(['supplier_segment', 'activity_type']).agg({
        'tour_id': 'count',
        'num_core_features': lambda x: (x > 0).sum()
    }).reset_index()
    combo['activation_rate'] = 100 * combo['num_core_features'] / combo['tour_id']
    combo = combo.nlargest(8, 'tour_id').sort_values('tour_id', ascending=True)
    combo['label'] = combo['supplier_segment'].str[:15] + ' ×\n' + combo['activity_type'].str[:15]
    
    bars = ax6.barh(range(len(combo)), combo['tour_id'],
                   color=tier_color, alpha=0.85, edgecolor='#424242', linewidth=1.2)
    ax6.set_yticks(range(len(combo)))
    ax6.set_yticklabels(combo['label'], fontsize=8)
    ax6.set_xlabel('Tours (Reach)', fontweight='bold', fontsize=11)
    ax6.set_title('Top Segment × Activity', fontweight='bold', fontsize=13, pad=12)
    ax6.grid(axis='x', alpha=0.25)
    
    for i, (_, row) in enumerate(combo.iterrows()):
        ax6.text(row['tour_id'] + row['tour_id']*0.02, i, 
                f"{row['tour_id']:,}\n({row['activation_rate']:.0f}%)",
                va='center', fontsize=8, fontweight='bold')
    
    # Chart 4.2: Individual Category Adoption
    ax7 = fig.add_subplot(gs[3, 1])
    ind_adoption = {}
    for col, name in INDIVIDUAL_CATEGORIES.items():
        if col in tier_df.columns:
            ind_adoption[name] = 100 * tier_df[col].sum() / tier_df['has_individual_pricing'].sum()
    
    sorted_ind = sorted(ind_adoption.items(), key=lambda x: x[1], reverse=True)
    ind_names = [x[0] for x in sorted_ind]
    ind_rates = [x[1] for x in sorted_ind]
    
    bars = ax7.barh(range(len(ind_names)), ind_rates,
                   color=['#FFECB3', '#FFD54F', '#FFC107', '#FF8F00', '#E65100'][:len(ind_names)],
                   alpha=0.85, edgecolor='#424242', linewidth=1.2)
    ax7.set_yticks(range(len(ind_names)))
    ax7.set_yticklabels(ind_names, fontsize=10)
    ax7.set_xlabel('% of Individual Pricing Tours', fontweight='bold', fontsize=11)
    ax7.set_title('Individual Categories', fontweight='bold', fontsize=13, pad=12)
    ax7.grid(axis='x', alpha=0.25)
    
    for i, (name, rate) in enumerate(sorted_ind):
        ax7.text(rate + max(ind_rates)*0.02, i, f'{rate:.1f}%', 
                va='center', fontsize=9, fontweight='bold')
    
    # Chart 4.3: Summary Table
    ax8 = fig.add_subplot(gs[3, 2])
    ax8.axis('off')
    
    summary_stats = [
        ['Tours (Reach)', f'{len(tier_df):,}'],
        ['Suppliers', f'{tier_df["supplier_id"].nunique():,}'],
        ['Total GMV', f'€{tier_df["gmv_l12mo"].sum()/1e6:.1f}M'],
        ['Avg Tours/Supplier', f'{len(tier_df)/tier_df["supplier_id"].nunique():.1f}'],
        ['Overall Activation', f'{100*(tier_df["num_core_features"]>0).sum()/len(tier_df):.1f}%'],
        ['', ''],
        ['Basic Only', f'{(tier_df["num_core_features"]==0).sum():,} ({100*(tier_df["num_core_features"]==0).sum()/len(tier_df):.1f}%)'],
        ['Power Users (3+)', f'{(tier_df["num_core_features"]>=3).sum():,} ({100*(tier_df["num_core_features"]>=3).sum()/len(tier_df):.1f}%)'],
        ['Managed Tours', f'{tier_df["is_managed"].sum():,} ({100*tier_df["is_managed"].sum()/len(tier_df):.1f}%)']
    ]
    
    table = ax8.table(cellText=summary_stats, cellLoc='left', loc='center',
                     colWidths=[0.55, 0.45], bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.8)
    
    for i in range(len(summary_stats)):
        if i == 5:
            continue
        table[(i, 0)].set_facecolor('#E3F2FD')
        table[(i, 0)].set_text_props(weight='bold')
        table[(i, 1)].set_facecolor('#FAFAFA')
        table[(i, 1)].set_text_props(weight='bold', fontsize=11)
    
    ax8.set_title(f'{tier}\nSUMMARY', fontweight='bold', fontsize=14, pad=25, y=0.97)
    
    plt.tight_layout()
    plt.show()
    
    print(f"  ✓ {tier} complete")


In [0]:
# ============================================================================
# STEP 4: GRANULAR SEGMENT × ACTIVITY BREAKDOWN
# ============================================================================

print("\n[4/5] Creating granular segment × activity analysis...")

for tier_idx, tier in enumerate(GMV_TIERS):
    tier_df = df[df['gmv_tier'] == tier].copy()
    
    if len(tier_df) == 0:
        continue
    
    top_5_segments = tier_df['supplier_segment'].value_counts().head(5).index.tolist()
    
    for seg_idx, segment in enumerate(top_5_segments):
        seg_df = tier_df[tier_df['supplier_segment'] == segment].copy()
        
        if len(seg_df) < 20:
            continue
        
        print(f"\n  {tier} → {segment}: {len(seg_df):,} tours")
        
        fig = plt.figure(figsize=(22, 13))
        gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35, top=0.92, bottom=0.08)
        
        fig.suptitle(f'{tier.upper()} → {segment.upper()}\nREACH: {len(seg_df):,} tours | {seg_df["supplier_id"].nunique():,} suppliers | €{seg_df["gmv_l12mo"].sum()/1e6:.2f}M GMV',
                    fontsize=18, fontweight='bold', y=0.96)
        
        # Chart 1: Feature Activation
        ax1 = fig.add_subplot(gs[0, 0])
        seg_activation = {}
        for col, name in PRICING_FEATURES.items():
            if col in seg_df.columns:
                seg_activation[name] = 100 * seg_df[col].sum() / len(seg_df)
        
        sorted_seg = sorted(seg_activation.items(), key=lambda x: x[1], reverse=True)
        seg_names = [x[0] for x in sorted_seg]
        seg_rates = [x[1] for x in sorted_seg]
        
        bars = ax1.barh(range(len(seg_names)), seg_rates,
                       color=COLORS['features'][:len(seg_names)], alpha=0.85,
                       edgecolor='#424242', linewidth=1.2)
        ax1.set_yticks(range(len(seg_names)))
        ax1.set_yticklabels(seg_names, fontsize=10)
        ax1.set_xlabel('Activation Rate (%)', fontweight='bold', fontsize=11)
        ax1.set_title('ACTIVATION', fontweight='bold', fontsize=13, pad=12)
        ax1.grid(axis='x', alpha=0.25)
        
        for i, (name, rate) in enumerate(sorted_seg):
            tours = seg_df[[k for k, v in PRICING_FEATURES.items() if v == name][0]].sum()
            ax1.text(rate + max(seg_rates)*0.02, i, f'{rate:.1f}%\n({tours:,})',
                    va='center', fontsize=9, fontweight='bold')
        
        # Chart 2: Activity Distribution
        ax2 = fig.add_subplot(gs[0, 1])
        act_dist = seg_df['activity_type'].value_counts().head(8).sort_values(ascending=True)
        bars = ax2.barh(range(len(act_dist)), act_dist.values,
                       color=COLORS['features'][::-1][:len(act_dist)], alpha=0.85,
                       edgecolor='#424242', linewidth=1.2)
        ax2.set_yticks(range(len(act_dist)))
        ax2.set_yticklabels(act_dist.index, fontsize=9)
        ax2.set_xlabel('Tours (Reach)', fontweight='bold', fontsize=11)
        ax2.set_title('Activity Distribution', fontweight='bold', fontsize=13, pad=12)
        ax2.grid(axis='x', alpha=0.25)
        
        for i, count in enumerate(act_dist.values):
            pct = 100 * count / len(seg_df)
            ax2.text(count + count*0.02, i, f'{count:,}\n({pct:.1f}%)',
                    va='center', fontsize=8, fontweight='bold')
        
        # Chart 3: Sophistication
        ax3 = fig.add_subplot(gs[0, 2])
        soph = seg_df['feature_sophistication'].value_counts().reindex([
            'Basic (Individual Only)',
            'Single Advanced Feature',
            'Multi-Feature',
            'Advanced',
            'Power User'
        ], fill_value=0)
        
        colors_soph = ['#BDBDBD', '#FFE0B2', '#FFB74D', '#FF9800', '#E65100']
        bars = ax3.bar(range(len(soph)), soph.values,
                      color=colors_soph, alpha=0.85, edgecolor='#424242', linewidth=1.2)
        ax3.set_xticks(range(len(soph)))
        ax3.set_xticklabels(['Basic', 'Single', 'Multi', 'Adv', 'Power'], fontsize=9)
        ax3.set_ylabel('Tours', fontweight='bold', fontsize=11)
        ax3.set_title('ENGAGEMENT DEPTH', fontweight='bold', fontsize=13, pad=12)
        ax3.grid(axis='y', alpha=0.25)
        
        for i, count in enumerate(soph.values):
            pct = 100 * count / len(seg_df)
            ax3.text(i, count + count*0.02, f'{count:,}\n({pct:.0f}%)',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')
        
        # Chart 4-9: Activation by Activity Type Heatmap
        ax4 = fig.add_subplot(gs[1:, :])
        top_acts = seg_df['activity_type'].value_counts().head(10).index.tolist()
        heatmap_data = []
        act_labels = []
        for act in top_acts:
            act_df = seg_df[seg_df['activity_type'] == act]
            row = []
            for col in PRICING_FEATURES.keys():
                if col in act_df.columns:
                    adoption = 100 * act_df[col].sum() / len(act_df) if len(act_df) > 0 else 0
                    row.append(adoption)
            heatmap_data.append(row)
            act_labels.append(f"{act}\n(n={len(act_df):,})")
        
        heatmap_df = pd.DataFrame(heatmap_data, columns=list(PRICING_FEATURES.values()), index=act_labels)
        sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax4,
                    cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
                    vmin=0, vmax=80, annot_kws={'fontsize': 10, 'weight': 'bold'})
        ax4.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
        ax4.set_ylabel('Activity Type', fontweight='bold', fontsize=13)
        ax4.set_title('ACTIVATION BY ACTIVITY TYPE (% of tours in activity)', fontweight='bold', fontsize=15, pad=18)
        ax4.set_yticklabels(act_labels, rotation=0, fontsize=10)
        ax4.set_xticklabels(list(PRICING_FEATURES.values()), rotation=35, ha='right', fontsize=11)
        
        plt.tight_layout()
        plt.show()
    
    print(f"  ✓ Granular analysis for {tier} complete")


In [0]:
# ============================================================================
# STEP 5: CROSS-TIER COMPARISON SUMMARY
# ============================================================================

print("\n[5/5] Creating cross-tier comparison...")

fig = plt.figure(figsize=(24, 14))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35, top=0.93, bottom=0.07)

fig.suptitle('CROSS-TIER COMPARISON - PRICING FEATURE ADOPTION',
             fontsize=22, fontweight='bold', y=0.97)

# Chart 1: Activation Comparison
ax1 = fig.add_subplot(gs[0, :])
comparison_data = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in tier_df.columns:
            adoption = 100 * tier_df[col].sum() / len(tier_df) if len(tier_df) > 0 else 0
            row.append(adoption)
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data, columns=list(PRICING_FEATURES.values()), index=GMV_TIERS)

x = np.arange(len(GMV_TIERS))
width = 0.11
feature_list = list(PRICING_FEATURES.values())

for i, feature in enumerate(feature_list):
    offset = width * (i - len(feature_list)/2 + 0.5)
    bars = ax1.bar(x + offset, comparison_df[feature], width, 
                  label=feature, color=COLORS['features'][i], alpha=0.85,
                  edgecolor='#424242', linewidth=1.2)

ax1.set_xlabel('GMV Tier', fontweight='bold', fontsize=13)
ax1.set_ylabel('Activation Rate (%)', fontweight='bold', fontsize=13)
ax1.set_title('ACTIVATION COMPARISON ACROSS GMV TIERS', fontweight='bold', fontsize=16, pad=15)
ax1.set_xticks(x)
ax1.set_xticklabels(['Bottom 25%', '25-50%', '50-75%', 'Top 25%'], fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', fontsize=11, ncol=2, framealpha=0.95)
ax1.grid(axis='y', alpha=0.25)

# Chart 2: Average Engagement Depth
ax2 = fig.add_subplot(gs[1, 0])
avg_features = df.groupby('gmv_tier')['num_core_features'].mean().reindex(GMV_TIERS)
bars = ax2.bar(range(len(avg_features)), avg_features.values,
              color=COLORS['gmv_tiers'], alpha=0.85, edgecolor='#424242', linewidth=1.5)
ax2.set_xticks(range(len(avg_features)))
ax2.set_xticklabels(['Bottom 25%', '25-50%', '50-75%', 'Top 25%'], fontsize=11, fontweight='bold')
ax2.set_ylabel('Avg Advanced Features', fontweight='bold', fontsize=12)
ax2.set_title('ENGAGEMENT DEPTH by Tier', fontweight='bold', fontsize=14, pad=12)
ax2.grid(axis='y', alpha=0.25)

for i, val in enumerate(avg_features.values):
    ax2.text(i, val + 0.03, f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Chart 3: Power Users %
ax3 = fig.add_subplot(gs[1, 1])
power_users = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    pct = 100 * (tier_df['num_core_features'] >= 3).sum() / len(tier_df) if len(tier_df) > 0 else 0
    power_users.append(pct)

bars = ax3.bar(range(len(power_users)), power_users,
              color=COLORS['gmv_tiers'], alpha=0.85, edgecolor='#424242', linewidth=1.5)
ax3.set_xticks(range(len(GMV_TIERS)))
ax3.set_xticklabels(['Bottom 25%', '25-50%', '50-75%', 'Top 25%'], fontsize=11, fontweight='bold')
ax3.set_ylabel('% of Tours', fontweight='bold', fontsize=12)
ax3.set_title('POWER USERS (3+ Features)', fontweight='bold', fontsize=14, pad=12)
ax3.grid(axis='y', alpha=0.25)

for i, val in enumerate(power_users):
    ax3.text(i, val + 0.3, f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Chart 4: Non-Adopters %
ax4 = fig.add_subplot(gs[1, 2])
non_adopters = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    pct = 100 * (tier_df['num_core_features'] == 0).sum() / len(tier_df) if len(tier_df) > 0 else 0
    non_adopters.append(pct)

bars = ax4.bar(range(len(non_adopters)), non_adopters,
              color=COLORS['gmv_tiers'], alpha=0.85, edgecolor='#424242', linewidth=1.5)
ax4.set_xticks(range(len(GMV_TIERS)))
ax4.set_xticklabels(['Bottom 25%', '25-50%', '50-75%', 'Top 25%'], fontsize=11, fontweight='bold')
ax4.set_ylabel('% of Tours', fontweight='bold', fontsize=12)
ax4.set_title('NON-ADOPTERS (Basic Only)', fontweight='bold', fontsize=14, pad=12)
ax4.grid(axis='y', alpha=0.25)

for i, val in enumerate(non_adopters):
    ax4.text(i, val + 0.5, f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Chart 5: Summary Table
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('off')

summary_table = []
summary_table.append(['METRIC', 'BOTTOM 25%', '25-50%', '50-75%', 'TOP 25%', 'OVERALL'])

for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]

metrics = ['Reach (Tours)', 'Reach (Suppliers)', 'Total GMV', 'Avg Features', 'Overall Activation',
           'Power Users %', 'Non-Adopters %', 'Managed %']

for metric in metrics:
    row = [metric]
    for tier in GMV_TIERS:
        tier_df = df[df['gmv_tier'] == tier]
        if metric == 'Reach (Tours)':
            row.append(f'{len(tier_df):,}')
        elif metric == 'Reach (Suppliers)':
            row.append(f'{tier_df["supplier_id"].nunique():,}')
        elif metric == 'Total GMV':
            row.append(f'€{tier_df["gmv_l12mo"].sum()/1e6:.1f}M')
        elif metric == 'Avg Features':
            row.append(f'{tier_df["num_core_features"].mean():.2f}')
        elif metric == 'Overall Activation':
            row.append(f'{100*(tier_df["num_core_features"]>0).sum()/len(tier_df):.1f}%')
        elif metric == 'Power Users %':
            row.append(f'{100*(tier_df["num_core_features"]>=3).sum()/len(tier_df):.1f}%')
        elif metric == 'Non-Adopters %':
            row.append(f'{100*(tier_df["num_core_features"]==0).sum()/len(tier_df):.1f}%')
        elif metric == 'Managed %':
            row.append(f'{100*tier_df["is_managed"].sum()/len(tier_df):.1f}%')
    
    # Overall column
    if metric == 'Reach (Tours)':
        row.append(f'{len(df):,}')
    elif metric == 'Reach (Suppliers)':
        row.append(f'{df["supplier_id"].nunique():,}')
    elif metric == 'Total GMV':
        row.append(f'€{df["gmv_l12mo"].sum()/1e6:.1f}M')
    elif metric == 'Avg Features':
        row.append(f'{df["num_core_features"].mean():.2f}')
    elif metric == 'Overall Activation':
        row.append(f'{100*(df["num_core_features"]>0).sum()/len(df):.1f}%')
    elif metric == 'Power Users %':
        row.append(f'{100*(df["num_core_features"]>=3).sum()/len(df):.1f}%')
    elif metric == 'Non-Adopters %':
        row.append(f'{100*(df["num_core_features"]==0).sum()/len(df):.1f}%')
    elif metric == 'Managed %':
        row.append(f'{100*df["is_managed"].sum()/len(df):.1f}%')
    
    summary_table.append(row)

table = ax5.table(cellText=summary_table, cellLoc='center', loc='center',
                 colWidths=[0.18, 0.14, 0.14, 0.14, 0.14, 0.14],
                 bbox=[0.02, 0.1, 0.96, 0.85])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 3.2)

# Header row
for i in range(6):
    table[(0, i)].set_facecolor('#1565C0')
    table[(0, i)].set_text_props(weight='bold', color='white', fontsize=11)

# Data rows
for i in range(1, len(summary_table)):
    table[(i, 0)].set_facecolor('#E3F2FD')
    table[(i, 0)].set_text_props(weight='bold')
    for j in range(1, 6):
        if j == 5:
            table[(i, j)].set_facecolor('#FFF3E0')
            table[(i, j)].set_text_props(weight='bold')
        else:
            tier_idx = j - 1
            table[(i, j)].set_facecolor(['#E3F2FD', '#BBDEFB', '#90CAF9', '#64B5F6'][tier_idx])

ax5.set_title('COMPREHENSIVE SUMMARY TABLE', fontweight='bold', fontsize=18, pad=35, y=0.96)

plt.tight_layout()
plt.show()

print("\n" + "="*100)
print("✓ PRICING FEATURES ADOPTION BASELINE ANALYSIS COMPLETE")
print("="*100)
print("\nSUMMARY:")
print(f"  Total Tours: {len(df):,}")
print(f"  Total Suppliers: {df['supplier_id'].nunique():,}")
print(f"  Overall Activation: {100*(df['num_core_features']>0).sum()/len(df):.1f}%")
print(f"  Power Users (3+ features): {100*(df['num_core_features']>=3).sum()/len(df):.1f}%")
print("="*100)
